# Informe Técnico: Sistema de Recuperación de Información

**Autor:** David Esteban Morales  
**Materia:** Recuperación de Información — Prof. Iván Carrera  
**Fecha:** 01 de junio de 2026

---

## 1. Descripción del Corpus

Se utilizó el dataset **Reuters "Uncovering Financial Insights with the Reuters 2"** disponible en Kaggle, archivo `ModApte_train.csv`. Este corpus contiene aproximadamente **9,000 documentos financieros** con las siguientes características:

- **Dominio:** Noticias financieras (mercados, commodities, política monetaria, comercio)
- **Campos utilizados:** `title` + `text` (concatenados como texto crudo de cada documento)
- **Etiquetas:** Columna `topics` con categorías como *earn*, *acq*, *grain*, *trade*, etc.
- **Uso de topics:** Se emplean como juicios de relevancia (qrels) para la evaluación automática

### Carga y exploración inicial

In [ ]:
import kagglehub, os, pandas as pd

path = kagglehub.dataset_download("thedevastator/uncovering-financial-insights-with-the-reuters-2")
csv_path = os.path.join(path, 'ModApte_train.csv')
df = pd.read_csv(csv_path)
print(f"Documentos: {len(df)}")
print(f"Columnas: {list(df.columns)}")
df[['title', 'text', 'topics']].head(3)

## 2. Decisiones de Diseño

### 2.1 Preprocesamiento

Se implementó un pipeline de texto (`preprocessing.py`) con las siguientes etapas:

1. **Limpieza:** eliminación de caracteres no alfabéticos con `re.sub(r'[^a-z\s]', '', text)`
2. **Tokenización:** NLTK `word_tokenize`
3. **Stopwords:** remoción de stopwords en inglés (NLTK)
4. **Stemming:** `SnowballStemmer('english')`

Se utiliza **caché con joblib** para persistir el corpus procesado y el índice invertido, evitando reprocesamiento en ejecuciones posteriores.

### 2.2 Índice Invertido

Estructura: diccionario `{término: {doc_id: frecuencia}}` usando `defaultdict`. Todos los modelos clásicos (Jaccard, TF-IDF, BM25) operan sobre este índice, justificando su construcción.

### 2.3 Arquitectura de Modelos

Se definió una clase abstracta `RetrievalModel` con método `search()`, permitiendo polimorfismo. Cada modelo está en su propio archivo dentro de `models/`:

| Modelo | Archivo | Características clave |
|--------|---------|----------------------|
| Jaccard | `jaccard.py` | Similitud de conjuntos binaria; precomputa `doc_unique_terms` |
| TF-IDF | `tfidf.py` | TF logarítmico `1+log(tf)`, IDF clásico `log(N/df)`, normalización coseno. Esquema lnc.ltc |
| BM25 | `bm25.py` | k1=1.5, b=0.75; penalización por longitud integrada; sin normalización coseno |
| Semántico | `semantic.py` | `all-MiniLM-L6-v2` (384 dims) + FAISS `IndexFlatIP`; query sin preprocesar |

### 2.4 Diferencias clave entre modelos

- **TF-IDF vs BM25:** TF-IDF normaliza con coseno y usa TF logarítmico; BM25 penaliza explícitamente por longitud del documento (`dl/avgdl`) sin coseno.
- **Clásicos vs Semántico:** Los modelos clásicos requieren coincidencia exacta de términos (post-stemming); el semántico captura relaciones de significado (sinónimos, paráfrasis) sin preprocesamiento de la query.

In [ ]:
# Demostración del preprocesamiento
import sys, re
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

stop_words = set(stopwords.words('english'))
stemmer = SnowballStemmer('english')

def process_raw(raw_text: str) -> list:
    clean_text = re.sub(r'[^a-z\s]', '', raw_text.lower())
    tokens = word_tokenize(clean_text)
    return [stemmer.stem(t) for t in tokens if t not in stop_words]

# Ejemplo
ejemplo = "Japanese Central Bank Interest Rate Policy"
print(f"Original: {ejemplo}")
print(f"Procesado: {process_raw(ejemplo)}")

## 3. Ejemplos de Consultas y Resultados

Se seleccionaron 4 queries que demuestran las fortalezas y debilidades de cada modelo:

1. **"earn"** — término exacto, alta frecuencia
2. **"oil prices rising"** — multi-token, BM25 debería destacar
3. **"chocolate commerce"** — sinónimos (cocoa≈chocolate), ventaja semántica
4. **"japanese central bank interest rate policy"** — query larga, Jaccard se diluye

In [ ]:
from collections import defaultdict, Counter
import numpy as np

# Procesar corpus
def process_corpus(path):
    corpus = {'doc_id': [], 'title': [], 'raw': [], 'tokenized': [], 'processed': []}
    df_input = pd.read_csv(path)
    for id_doc, row in df_input.iterrows():
        text = str(row.get('title','')) + ' ' + str(row.get('text',''))
        stemmed = process_raw(text)
        corpus['doc_id'].append(id_doc)
        corpus['title'].append(row.get('title',''))
        corpus['raw'].append(text)
        corpus['tokenized'].append(stemmed)
        corpus['processed'].append(' '.join(stemmed))
    return pd.DataFrame(corpus)

def build_inv_index(df_corpus):
    index = defaultdict(dict)
    for _, row in df_corpus.iterrows():
        doc_id = row['doc_id']
        conteo = Counter(row['tokenized'])
        for term, f in conteo.items():
            index[term][doc_id] = f
    return index

print("Procesando corpus...")
df_corpus = process_corpus(csv_path)
inv_index = build_inv_index(df_corpus)
print(f"Corpus: {len(df_corpus)} docs, Vocabulario: {len(inv_index)} términos")

In [ ]:
# Implementación de los 4 modelos

class JaccardModel:
    def __init__(self, df_corpus, inv_index):
        self.inv_index = inv_index
        self.N = len(df_corpus)
        self.doc_unique_terms = df_corpus['tokenized'].apply(lambda t: len(set(t))).values

    def search(self, query_processed, top_n=10):
        query_terms = set(query_processed.split())
        intersection = np.zeros(self.N)
        for term in query_terms:
            if term in self.inv_index:
                for doc_id in self.inv_index[term]:
                    intersection[doc_id] += 1
        union = self.doc_unique_terms + len(query_terms) - intersection
        scores = np.divide(intersection, union, out=np.zeros(self.N), where=union!=0)
        ranking = scores.argsort()[::-1]
        return ranking[:top_n], scores

class TFIDFModel:
    def __init__(self, df_corpus, inv_index):
        self.inv_index = inv_index
        self.N = len(df_corpus)
        self.doc_norms = self._precompute_norms()

    def _precompute_norms(self):
        doc_norms = np.zeros(self.N)
        for term, doc_freqs in self.inv_index.items():
            df_t = len(doc_freqs)
            idf = np.log(self.N / df_t)
            for doc_id, tf in doc_freqs.items():
                w = (1 + np.log(tf)) * idf
                doc_norms[doc_id] += w ** 2
        return np.sqrt(doc_norms)

    def search(self, query_processed, top_n=10):
        query_terms = query_processed.split()
        scores = np.zeros(self.N)
        query_norm_sq = 0.0
        for term in query_terms:
            if term not in self.inv_index:
                continue
            doc_freqs = self.inv_index[term]
            df_t = len(doc_freqs)
            idf = np.log(self.N / df_t)
            q_weight = (1 + np.log(query_terms.count(term))) * idf
            query_norm_sq += q_weight ** 2
            for doc_id, tf in doc_freqs.items():
                d_weight = (1 + np.log(tf)) * idf
                scores[doc_id] += d_weight * q_weight
        query_norm = np.sqrt(query_norm_sq)
        norm = self.doc_norms * query_norm
        scores = np.divide(scores, norm, out=np.zeros(self.N), where=norm!=0)
        ranking = scores.argsort()[::-1]
        return ranking[:top_n], scores

class BM25Model:
    def __init__(self, df_corpus, inv_index, k1=1.5, b=0.75):
        self.inv_index = inv_index
        self.N = len(df_corpus)
        self.k1, self.b = k1, b
        self.doc_lengths = df_corpus['tokenized'].apply(len).values
        self.avgdl = self.doc_lengths.mean()

    def search(self, query_processed, top_n=10):
        query_terms = query_processed.split()
        scores = np.zeros(self.N)
        for term in query_terms:
            if term not in self.inv_index:
                continue
            doc_freqs = self.inv_index[term]
            n_t = len(doc_freqs)
            idf = np.log((self.N - n_t + 0.5) / (n_t + 0.5) + 1)
            for doc_id, tf in doc_freqs.items():
                dl = self.doc_lengths[doc_id]
                num = tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * (dl / self.avgdl))
                scores[doc_id] += idf * (num / den)
        ranking = scores.argsort()[::-1]
        return ranking[:top_n], scores

print("Construyendo modelos clásicos...")
jaccard = JaccardModel(df_corpus, inv_index)
tfidf = TFIDFModel(df_corpus, inv_index)
bm25 = BM25Model(df_corpus, inv_index)
print("Modelos clásicos listos.")

In [ ]:
# Modelo semántico
from sentence_transformers import SentenceTransformer
import faiss

print("Cargando modelo de embeddings (all-MiniLM-L6-v2)...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generando embeddings del corpus...")
doc_embeddings = embed_model.encode(df_corpus['raw'].tolist(), show_progress_bar=True)
faiss.normalize_L2(doc_embeddings)

dimension = doc_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(doc_embeddings.astype('float32'))
print(f"Índice FAISS construido: {faiss_index.ntotal} vectores, {dimension} dimensiones")

def semantic_search(query_text, top_n=10):
    q_emb = embed_model.encode([query_text], normalize_embeddings=True)
    sim_scores, indices = faiss_index.search(q_emb.astype('float32'), top_n)
    N = faiss_index.ntotal
    scores = np.zeros(N)
    ranking = indices.flatten()
    for i, idx in enumerate(ranking):
        scores[idx] = sim_scores.flatten()[i]
    return ranking, scores

print("Modelo semántico listo.")

In [ ]:
# Comparación de resultados para las 4 queries
K = 10
queries = [
    "earn",
    "oil prices rising",
    "chocolate commerce",
    "japanese central bank interest rate policy"
]

modelos = {
    'Jaccard': (jaccard, False),
    'TF-IDF Coseno': (tfidf, False),
    'BM25': (bm25, False),
}

def run_query(query, top_n=5):
    """Ejecuta los 4 modelos y muestra top-5 resultados."""
    print(f"\n{'='*80}")
    print(f"QUERY: \"{query}\"")
    print(f"{'='*80}")
    
    for nombre, (modelo, es_sem) in modelos.items():
        if es_sem:
            q = query
        else:
            q = ' '.join(process_raw(query))
        ranking, scores = modelo.search(q, top_n=top_n)
        print(f"\n  [{nombre}]")
        for i, idx in enumerate(ranking[:top_n]):
            print(f"    {i+1}. (score={scores[idx]:.4f}) {df_corpus.loc[idx, 'title']}")
    
    # Semántico
    ranking, scores = semantic_search(query, top_n=top_n)
    print(f"\n  [Semántico]")
    for i, idx in enumerate(ranking[:top_n]):
        print(f"    {i+1}. (score={scores[idx]:.4f}) {df_corpus.loc[idx, 'title']}")

for q in queries:
    run_query(q, top_n=5)

## 4. Evaluación y Análisis de Métricas

### 4.1 Construcción de Qrels

Los juicios de relevancia se construyen a partir de la columna `topics` del dataset. Cada topic se usa como query de evaluación, y los documentos etiquetados con ese topic son los relevantes. Se filtran topics con menos de 10 documentos para tener evaluaciones estadísticamente significativas.

### 4.2 Métricas implementadas (desde cero, sin sklearn)

- **Precision@K:** proporción de documentos relevantes en el top-K
- **Recall@K:** proporción de documentos relevantes del total que aparecen en el top-K
- **Average Precision (AP):** premia que los relevantes aparezcan en posiciones altas
- **MAP:** promedio de AP sobre todas las queries

In [ ]:
# Métricas
def precision_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    hits = sum(1 for doc in retrieved[:k] if doc in relevant)
    return hits / k

def recall_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    if not relevant: return 0.0
    hits = sum(1 for doc in retrieved[:k] if doc in relevant)
    return hits / len(relevant)

def average_precision(retrieved, relevant):
    relevant = set(relevant)
    if not relevant: return 0.0
    suma, hits = 0.0, 0
    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            hits += 1
            suma += hits / i
    return suma / len(relevant)

# Cargar qrels
import re as re_mod
def parse_topics(raw):
    return re_mod.findall(r"'([^']+)'", str(raw))

def cargar_qrels(csv_path, min_docs=10):
    df_q = pd.read_csv(csv_path)
    qrels = defaultdict(list)
    for doc_id, row in df_q.iterrows():
        for topic in parse_topics(row['topics']):
            qrels[topic].append(doc_id)
    return {t: docs for t, docs in qrels.items() if len(docs) >= min_docs}

qrels = cargar_qrels(csv_path)
print(f"Queries de evaluación: {len(qrels)} topics")
print(f"Ejemplos: {list(qrels.keys())[:10]}")

In [ ]:
# Evaluación batch
def evaluar_modelo(retrieve_fn, qrels, k=10):
    per_query = {}
    for query, relevantes in qrels.items():
        ranking = retrieve_fn(query)
        doc_ids = [df_corpus.loc[idx, 'doc_id'] for idx in ranking]
        per_query[query] = {
            'P@K': precision_at_k(doc_ids, relevantes, k),
            'R@K': recall_at_k(doc_ids, relevantes, k),
            'AP': average_precision(doc_ids, relevantes),
        }
    n = len(per_query)
    return {
        'MAP': sum(m['AP'] for m in per_query.values()) / n,
        'avg P@10': sum(m['P@K'] for m in per_query.values()) / n,
        'avg R@10': sum(m['R@K'] for m in per_query.values()) / n,
    }

print("Ejecutando evaluación batch...\n")

resultados = {}

# Jaccard
resultados['Jaccard'] = evaluar_modelo(
    lambda q: jaccard.search(' '.join(process_raw(q)), top_n=K)[0], qrels, K)

# TF-IDF
resultados['TF-IDF Coseno'] = evaluar_modelo(
    lambda q: tfidf.search(' '.join(process_raw(q)), top_n=K)[0], qrels, K)

# BM25
resultados['BM25'] = evaluar_modelo(
    lambda q: bm25.search(' '.join(process_raw(q)), top_n=K)[0], qrels, K)

# Semántico
resultados['Semántico'] = evaluar_modelo(
    lambda q: semantic_search(q, top_n=K)[0], qrels, K)

# Tabla de resultados
print(f"{'Modelo':<16} {'MAP':>8} {'avg P@10':>10} {'avg R@10':>10}")
print("-" * 46)
for modelo, metricas in resultados.items():
    print(f"{modelo:<16} {metricas['MAP']:>8.4f} {metricas['avg P@10']:>10.4f} {metricas['avg R@10']:>10.4f}")

### 4.3 Resultados y Análisis

| Modelo | MAP | avg P@10 | avg R@10 |
|--------|-----|----------|----------|
| Jaccard | 0.0541 | 0.2246 | 0.0628 |
| TF-IDF Coseno | 0.0632 | 0.2557 | 0.0723 |
| BM25 | 0.0666 | 0.2525 | 0.0742 |
| Semántico | 0.0801 | 0.3115 | 0.0861 |

**Orden obtenido:** Jaccard < TF-IDF < BM25 < Semántico

### 4.4 Análisis por tipo de consulta

- **Queries de término único ("earn"):** Todos los modelos rinden similar porque no hay ambigüedad léxica. BM25 y TF-IDF producen rankings casi idénticos.

- **Queries multi-token ("oil prices rising"):** BM25 supera a TF-IDF gracias a su penalización por longitud de documento, evitando que documentos muy largos dominen solo por acumular términos.

- **Queries con sinónimos ("chocolate commerce"):** El modelo semántico destaca significativamente. Los modelos clásicos no encuentran documentos sobre "cocoa" porque el stemming no relaciona "chocolate" con "cocoa". El modelo de embeddings sí captura esta relación semántica.

- **Queries largas ("japanese central bank interest rate policy"):** Jaccard se diluye porque su similitud de conjuntos penaliza cuando `|A ∪ B|` crece. BM25 maneja mejor las queries largas al ponderar cada término independientemente.

### 4.5 Nota sobre Recall bajo

El recall aparece bajo (~6-8%) porque topics como "earn" tienen 2,877 documentos relevantes y solo evaluamos top-10. Esto es esperado y correcto — no se modificó el denominador de recall.

---

## 5. Conclusiones

1. **El orden de rendimiento** Jaccard < TF-IDF < BM25 < Semántico es consistente con la teoría de RI.
2. **BM25** es el mejor modelo clásico gracias a su normalización por longitud y saturación de frecuencia.
3. **La recuperación semántica** muestra su mayor ventaja en queries donde hay variación léxica (sinónimos, paráfrasis).
4. **Los modelos clásicos** siguen siendo competitivos para queries literales donde no hay gap semántico.
5. **MAP** es la métrica más informativa para este corpus dada la distribución desigual de documentos relevantes por topic.